# Промпт для оценки релевантности музыкальной выдачи (GigaChat-Pro)

Тестовое задание на промпт-инженера. Модель получает запрос пользователя, название трека и исполнителя из поисковой выдачи и ставит одну из четырёх меток: `vital`, `rel_plus`, `rel_minus`, `notrel`.

Валидационный набор `valid_set.tsv` (105 примеров) в репозиторий не входит. Ключ GigaChat берётся из переменной окружения `GIGACHAT_CREDENTIALS`.

In [4]:
# валидационный сет для разработки и дальнейшей корректировки промпта
import pandas as pd

test = pd.read_csv('valid_set.tsv', sep='\t')
print(f'Размер сета: {len(test)} строк')
print('\nРаспределение классов:')
print(test['golden_answer'].value_counts())


Размер сета: 105 строк

Распределение классов:
golden_answer
rel_plus     39
vital        28
notrel       22
rel_minus    16
Name: count, dtype: int64


In [ ]:
test.sample()

# **<span style="background-color: #fff69d">Предварительный анализ данных (EDA)</span>**

Прежде чем писать промпт — смотрим на данные. Цель: понять, какие сигналы
различают классы, и заложить эти сигналы в формулировки system_prompt.


In [ ]:
# 1. Распределение классов
import matplotlib.pyplot as plt

class_counts = test['golden_answer'].value_counts().reindex(
    ['vital', 'rel_plus', 'rel_minus', 'notrel']
)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(class_counts.index, class_counts.values,
              color=['#2ecc71', '#27ae60', '#f39c12', '#e74c3c'])
ax.set_title('Распределение классов в валидационном сете')
ax.set_ylabel('Количество примеров')
for bar, val in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.5,
            f'{val}\n({val / len(test):.0%})',
            ha='center', va='bottom', fontsize=10)
ax.set_ylim(0, max(class_counts.values) * 1.2)
plt.tight_layout()
plt.show()

print(f'Всего: {len(test)} примеров')
print(f'Дисбаланс: мажоритарный класс {class_counts.max() / len(test):.1%}, '
      f'миноритарный {class_counts.min() / len(test):.1%}')


**Вывод по распределению:**
Сет несбалансированный: `rel_plus` (37%) почти в 2.5 раза больше `rel_minus` (15%).
Это значит:
- **accuracy сама по себе обманет** — модель, всегда отвечающая `rel_plus`, получит ~37%.
- Главной метрикой должен быть **macro F1** (все классы важны одинаково).
- Маленький `rel_minus` — самый рискованный класс с точки зрения замеров: одна-две ошибки на нём роняют F1 заметно. На него нужно дать чёткие критерии в промпте.


In [ ]:
# 2. Длина запроса: важна только для vital
# (для остальных классов разброс одинаковый, отдельная группировка только зашумляет анализ)
test['n_words'] = test['query'].str.split().str.len()

vital_words = test[test['golden_answer'] == 'vital']['n_words']
other_words = test[test['golden_answer'] != 'vital']['n_words']

fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([vital_words, other_words], tick_labels=['vital', 'остальные'])
ax.set_title('Длина запроса (число слов): vital vs остальные классы')
ax.set_ylabel('слов в запросе')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'vital:     медиана {vital_words.median():.1f}, среднее {vital_words.mean():.2f}')
print(f'остальные: медиана {other_words.median():.1f}, среднее {other_words.mean():.2f}')


**Вывод по длине:**
`vital` имеет заметно более длинные запросы (медиана ~4.5 слов против ~3 у остальных).
Логика простая: чтобы получить оценку `vital`, пользователь обычно указывает **и трек, и исполнителя** одновременно — это и даёт больше слов в запросе.

У остальных классов (`rel_plus` / `rel_minus` / `notrel`) распределение длины примерно одинаковое, поэтому длина как признак между ними не различает. Отдельная группировка по этим трём классам только зашумляет анализ — поэтому смотрим только на различие vital vs остальные.

**Что это значит для промпта:** наличие **обоих** маркеров (трек + исполнитель) в запросе — сильный сигнал в пользу `vital`. Это попало в первый пункт критериев.


In [ ]:
# 3. Самый важный сигнал: совпадает ли найденный трек/автор с запросом
def normalize(s):
    return ''.join(c.lower() for c in str(s) if c.isalnum())

test['track_in_query'] = test.apply(
    lambda r: normalize(r['track_name'])[:6] in normalize(r['query'])
    if len(normalize(r['track_name'])) >= 3 else False, axis=1)
test['author_in_query'] = test.apply(
    lambda r: normalize(r['author'])[:5] in normalize(r['query'])
    if len(normalize(r['author'])) >= 3 else False, axis=1)

signal = test.groupby('golden_answer')[['track_in_query', 'author_in_query']].mean().reindex(order)

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(signal.index))
width = 0.35
ax.bar([i - width/2 for i in x], signal['track_in_query'], width,
       label='Название трека встречается в запросе', color='#3498db')
ax.bar([i + width/2 for i in x], signal['author_in_query'], width,
       label='Имя исполнителя встречается в запросе', color='#9b59b6')
ax.set_xticks(list(x))
ax.set_xticklabels(signal.index)
ax.set_ylabel('Доля примеров класса')
ax.set_title('Совпадение найденного трека/автора с текстом запроса (по подстроке)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print(signal.round(2))


**Вывод по совпадениям — самое важное:**
- В **82%** случаев класса `vital` название трека из выдачи буквально присутствует в тексте запроса. У `rel_minus` — только **19%**.
- По исполнителю разница ещё резче: `vital` — **57%**, `rel_minus` — **6%**.
- У `notrel` траки иногда совпадают по названию (45%!), но **по смыслу** запрос про другое. Это классический ложноположительный сигнал.

**Главный инсайт:** простое строковое совпадение — сильный, но недостаточный признак. Модели нужно научиться отличать:
- *настоящее совпадение по смыслу* («козина» → нужен Козин-певец, а не «Cozinha» — `rel_minus`);
- *коллизию популярности* (есть более известная одноимённая песня — тоже `rel_minus`);
- *смысловое попадание без буквального совпадения* («песня для сна» → «Песенка для сна» — `rel_plus`);
- *полное промахивание мимо темы* (`notrel`).

**Что из этого попало в промпт:**
1. Явный критерий для `vital`: трек и исполнитель совпадают одновременно.
2. Отдельный критерий для `rel_minus`: «существует более популярная широко известная песня с похожим названием» — на это нацелены примеры с «Козина» / «Маша».
3. Критерий для `notrel`: «найден совершенно другой трек/исполнитель, не упомянутый и не подразумеваемый в запросе» — для случаев, когда есть совпадение по подстроке, но смысл не тот.
4. Few-shot примеры подобраны так, чтобы покрыть каждый из этих случаев.


In [ ]:
# 4. Пограничные кейсы — что должно быть в промпте через примеры
print('=' * 70)
print('VITAL — есть трек + исполнитель, выдача точно та')
print('=' * 70)
for _, r in test[test['golden_answer'] == 'vital'].sample(3, random_state=1).iterrows():
    print(f"  query: {r['query']!r}")
    print(f"  track: {r['track_name']!r} | author: {r['author']!r}\n")

print('=' * 70)
print('REL_MINUS — выдача похожа, но рядом есть более популярный кандидат')
print('=' * 70)
for _, r in test[test['golden_answer'] == 'rel_minus'].sample(3, random_state=1).iterrows():
    print(f"  query: {r['query']!r}")
    print(f"  track: {r['track_name']!r} | author: {r['author']!r}\n")

print('=' * 70)
print('NOTREL — частые ложные совпадения по подстроке')
print('=' * 70)
for _, r in test[test['golden_answer'] == 'notrel'].sample(3, random_state=1).iterrows():
    print(f"  query: {r['query']!r}")
    print(f"  track: {r['track_name']!r} | author: {r['author']!r}\n")


### Итог EDA → решения по промпту

| Наблюдение в данных | Решение в промпте |
|---|---|
| Дисбаланс классов (`rel_plus` x2.5 от `rel_minus`) | Основная метрика — **macro F1**, а не accuracy |
| `vital` коррелирует с длиной запроса и наличием и трека, и автора | В критериях `vital` явно прописано: «в запросе указан и трек, и исполнитель — оба совпадают с выдачей» |
| `rel_minus` — это коллизии похожих названий и одноимённых популярных песен | Отдельный пункт «существует более популярная одноимённая песня» + примеры с «Козина», «Маша» |
| `notrel` часто имеет ложное совпадение по подстроке | Акцент в критериях: «не упомянутый **и не подразумеваемый** в запросе» |
| Запросы короткие, с опечатками и транслитом («мукка», «парор стелар») | Правило «учитывай возможные опечатки и транслит» |
| Часто встречается командное «включи»/«поставь» | Заложено в формат few-shot примеров |

После EDA становится понятно, **что именно** должно быть в system_prompt — и почему именно так, а не иначе.


In [ ]:
# требуется установить пакет для дальнейшего использования модели GigaChat
! pip install -U langchain-gigachat

In [ ]:
! pip install langchain-core

In [1]:
import os
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_gigachat import GigaChat

# Авторизация в GigaChat
# temperature=0 — задача классификационная, нужны воспроизводимые ответы.
# max_tokens=10 — метка занимает 1–3 токена, 10 с запасом. Страхует от того,
# что модель уйдёт в простыню, и экономит время.
llm = GigaChat(
    credentials=os.environ["GIGACHAT_CREDENTIALS"], #Ключ авторизации
    scope="GIGACHAT_API_PERS",
    model="GigaChat-Pro",
    # Отключает проверку наличия сертификатов НУЦ Минцифры
    verify_ssl_certs=False,
    streaming=False,
    temperature=0,
    max_tokens=10,
)


In [2]:
system_prompt = """
Ты — асессор музыкального поискового сервиса. На вход тебе подаются:
- запрос пользователя,
- название найденного трека,
- исполнитель найденного трека.

Твоя задача — оценить, насколько результат поиска соответствует запросу,
и выбрать РОВНО ОДНУ метку из четырёх:
vital, rel_plus, rel_minus, notrel.

КРИТЕРИИ
1) vital — пользователь нашёл именно то, что искал.
   - в запросе указаны и трек, и исполнитель — оба совпадают с выдачей;
   - запрос — точная цитата из текста песни с указанием исполнителя,
     и у этого исполнителя только одна такая песня;
   - запрос — саундтрек из конкретного фильма, и название фильма
     присутствует в названии найденного трека.

2) rel_plus — выдача полезная, пользователь, скорее всего, доволен.
   - запрошено название песни без исполнителя — найдена одноимённая песня;
   - запрос только по исполнителю — найдена его сольная песня
     (и нет более известного одноимённого исполнителя);
   - запрос по жанру/настроению/тематике — найдена песня в этом жанре;
   - запрос по тексту/смыслу — найдена песня с этим текстом
     или близким смыслом;
   - пользователь ошибся в имени соавтора, но очевидно хотел эту песню.

3) rel_minus — выдача похожа, но есть более вероятный кандидат,
   которого поиск не нашёл (см. чек-лист ниже).

4) notrel — выдача совсем не та, ответ бесполезен.

ВАЖНО: ПРЕЖДЕ ЧЕМ ПОСТАВИТЬ rel_plus — ПРОЙДИ ЧЕК-ЛИСТ

Модели свойственно завышать оценку и ставить rel_plus там, где правильно
было бы rel_minus или notrel. Чтобы этого избежать, перед каждым rel_plus
явно проверь следующие пункты. Если хотя бы один сработал — оценка ниже.

1. КОЛЛИЗИЯ ПОПУЛЯРНОСТИ = rel_minus.
    Запрос — короткое слово, фамилия или имя (1–2 слова)?
    Существует ли широко известный артист или хит с таким же
    или очень похожим написанием (включая опечатки и транслит)?
    Если ты можешь назвать такого известного кандидата, а в выдаче —
    другое (малоизвестный одноимённый трек, иностранное слово
    с похожим написанием) → ставь rel_minus.

    Известные кандидаты, на которых сервис часто промахивается:
    Вадим Козин, Маша Распутина, Михаил Розенбаум, Земфира,
    Parov Stelar, Валерий Ободзинский и т.п.

2.  КОЛЛАБ ВМЕСТО СОЛО = rel_minus.
    В запросе один исполнитель, а в выдаче — коллаб с другими,
    включая запрошенного? И у запрошенного есть много сольных хитов?
    Пользователь хотел сольник → ставь rel_minus.

3. ЧАСТИЧНОЕ СОВПАДЕНИЕ ПО ИСПОЛНИТЕЛЯМ = rel_minus.
    В запросе несколько исполнителей вместе, а в выдаче —
    только один из них без коллаборации с остальными → rel_minus.

4. ТЕМАТИКА НЕ ТА = rel_minus.
    Запрошен саундтрек/жанр/тематика, и в выдаче что-то с похожим
    словом, но другой жанр или другой источник? Например, запрос
    «титаник музыка» (саундтрек к фильму), а в выдаче песня
    «На Титанике» совсем другого автора → rel_minus.

5. ЛОЖНОЕ СОВПАДЕНИЕ ПО ПОДСТРОКЕ = notrel.
    Название трека из выдачи совпадает с запросом ТОЛЬКО по одному
    общему слову, но смысл запроса явно про другое (другой исполнитель,
    другая известная песня)?
    Пример: запрос «rammstein москва» — выдача «Москва» исполнителя Yanix.
    Слово совпало, Раммштайн ни при чём → notrel.

6. ИСПОЛНИТЕЛЬ/ПЕСНЯ НАЗВАНЫ, НО НЕ УГАДАНЫ = notrel.
    В запросе явно указан исполнитель и/или конкретная песня,
    а в выдаче ни один из них не угадан по смыслу → notrel,
    даже если есть совпадение по отдельным словам.

ПРИМЕРЫ
# vital — явное двойное совпадение
Запрос пользователя: цой звезда по имени солнце
Найденный трек: Звезда по имени Солнце
Исполнитель: Виктор Цой
Оценка: vital

# rel_plus — название без исполнителя
Запрос пользователя: поставь песню облака
Найденный трек: Облака
Исполнитель: Звери
Оценка: rel_plus

# rel_plus — только исполнитель, найдена его песня
Запрос пользователя: включи группу металлика
Найденный трек: Enter Sandman
Исполнитель: Metallica
Оценка: rel_plus

# rel_minus 1. — коллизия популярности
Запрос пользователя: включи козина
Найденный трек: Cozinha
Исполнитель: Volbru
Оценка: rel_minus

# rel_minus 1. — коллизия популярности
Запрос пользователя: маша музыка
Найденный трек: Маша
Исполнитель: Николай Анисимов
Оценка: rel_minus

# rel_minus 2. — коллаб вместо соло
Запрос пользователя: включи parov stelar
Найденный трек: Keine Melodien
Исполнитель: Jeans Team, Parov Stelar
Оценка: rel_minus

# rel_minus 3. — частичное совпадение исполнителей
Запрос пользователя: поставь клаву коку и моргенштерна
Найденный трек: Краш
Исполнитель: Клава Кока
Оценка: rel_minus

# rel_minus 4. — тематика не та
Запрос пользователя: титаник музыка
Найденный трек: На Титанике
Исполнитель: Лолита, INSTASAMKA
Оценка: rel_minus

# notrel 5. — ложное совпадение по подстроке
Запрос пользователя: включи rammstein москва
Найденный трек: Москва
Исполнитель: Yanix
Оценка: notrel

# notrel 6. — исполнитель/песня названы, но не угаданы
Запрос пользователя: хочешь артур пирожков клава кока
Найденный трек: ХОЧЕШЬ?
Исполнитель: Земфира
Оценка: notrel

ФОРМАТ ОТВЕТА
Верни ТОЛЬКО одну метку: vital, rel_plus, rel_minus или notrel.
Без объяснений, без кавычек, без знаков препинания.
""".strip()


In [5]:
# user_prompt формируется для каждой строки из (query, track_name, author)
def build_user_prompt(query: str, track_name: str, author: str) -> str:
    return (
        f"Запрос пользователя: {query}\n"
        f"Найденный трек: {track_name}\n"
        f"Исполнитель: {author}\n"
        f"Оценка:"
    )

# Пример
print(build_user_prompt(test.iloc[0]['query'],
                        test.iloc[0]['track_name'],
                        test.iloc[0]['author']))


Запрос пользователя: включи козина
Найденный трек: Cozinha
Исполнитель: Volbru
Оценка:


In [6]:
# Генерация ответов модели по всему валидационному сету
from tqdm import tqdm

raw_predictions = []  # сюда складываем сырые текстовые ответы модели

for _, row in tqdm(test.iterrows(), total=len(test)):
    user_prompt = build_user_prompt(row['query'], row['track_name'], row['author'])
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ]
    try:
        res = llm.invoke(messages)
        raw_predictions.append(res.content)
    except Exception as e:
        # На случай сетевых/квотных ошибок — не падаем, помечаем строку
        raw_predictions.append(f"__ERROR__: {e}")

test['raw_prediction'] = raw_predictions
test.head()


100%|██████████| 105/105 [01:34<00:00,  1.11it/s]


,query,track_name,author,golden_answer,raw_prediction
0,включи козина,Cozinha,Volbru,rel_minus,rel_minus
1,песня крокодил,Крокодил,"Словетский,DJ Nik One,OBLADAET,Джино",rel_plus,rel_plus
2,маша музыка,Маша,Николай Анисимов,rel_minus,rel_minus
3,включи песню алмазы на шее исполнитель топовский,Алмазы на шее,Топовский,vital,vital
4,мукка алиса,АЛИСА,МУККА,vital,vital


In [7]:
# Превью: запрос, выдача, эталон и сырой ответ модели
test[['query', 'track_name', 'author', 'golden_answer', 'raw_prediction']].head(10)


,query,track_name,author,golden_answer,raw_prediction
0,включи козина,Cozinha,Volbru,rel_minus,rel_minus
1,песня крокодил,Крокодил,"Словетский,DJ Nik One,OBLADAET,Джино",rel_plus,rel_plus
2,маша музыка,Маша,Николай Анисимов,rel_minus,rel_minus
3,включи песню алмазы на шее исполнитель топовский,Алмазы на шее,Топовский,vital,vital
4,мукка алиса,АЛИСА,МУККА,vital,vital
5,поставь клаву коку,РОЗОВАЯ ЛУНА,"Клава Кока,Джарахов,SQWOZ BAB",rel_minus,rel_plus
6,расскажи сказку про курочку рябу,Курочка Ряба,Мария Курская,vital,vital
7,поставь пожалуйста soraya bolero,Bolero (Hold Me In Your Arms) (Album Version),Soraya,vital,vital
8,включить песню для сна,Песенка для сна,Цветняшки,rel_plus,rel_plus
9,включи parov stelar,"Keine Melodien...1,2,3,4,5","Jeans Team,Parov Stelar",rel_minus,rel_minus


In [8]:
# Парсинг сырого ответа модели в одну из четырёх меток.
# Берём первую встретившуюся валидную метку в нижнем регистре.
import re

VALID_LABELS = ['vital', 'rel_plus', 'rel_minus', 'notrel']

def parse_label(raw: str) -> str:
    if not isinstance(raw, str):
        return 'unknown'
    text = raw.strip().lower()
    # Сначала ищем точное совпадение по словарю в порядке убывания "длины",
    # чтобы rel_plus / rel_minus не путались с rel.
    for label in sorted(VALID_LABELS, key=len, reverse=True):
        if re.search(r'\b' + re.escape(label) + r'\b', text):
            return label
    return 'unknown'

test['prediction'] = test['raw_prediction'].apply(parse_label)
print('Распределение предсказаний:')
print(test['prediction'].value_counts())
print('\nДоля нераспарсенных ответов:',
      (test['prediction'] == 'unknown').mean())


Распределение предсказаний:
prediction
rel_plus     45
vital        29
notrel       21
rel_minus    10
Name: count, dtype: int64

Доля нераспарсенных ответов: 0.0


In [9]:
# Метрики качества промпта
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
import pandas as pd

LABELS = ['vital', 'rel_plus', 'rel_minus', 'notrel']

y_true = test['golden_answer']
y_pred = test['prediction']

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, labels=LABELS, average='macro', zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, labels=LABELS, average='weighted', zero_division=0)

print(f'Accuracy:    {acc:.3f}')
print(f'Macro F1:    {macro_f1:.3f}')
print(f'Weighted F1: {weighted_f1:.3f}')

print('\nClassification report:')
print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0, digits=3))

print('\nConfusion matrix (строки — golden, столбцы — prediction):')
cm = confusion_matrix(y_true, y_pred, labels=LABELS)
print(pd.DataFrame(cm, index=LABELS, columns=LABELS))


Accuracy:    0.686
Macro F1:    0.644
Weighted F1: 0.677

Classification report:
              precision    recall  f1-score   support

       vital      0.828     0.857     0.842        28
    rel_plus      0.667     0.769     0.714        39
   rel_minus      0.600     0.375     0.462        16
      notrel      0.571     0.545     0.558        22

    accuracy                          0.686       105
   macro avg      0.666     0.637     0.644       105
weighted avg      0.679     0.686     0.677       105


Confusion matrix (строки — golden, столбцы — prediction):
           vital  rel_plus  rel_minus  notrel
vital         24         3          0       1
rel_plus       4        30          2       3
rel_minus      0         5          6       5
notrel         1         7          2      12


### Почему выбраны такие метрики

Задача — мультиклассовая классификация на 4 классa с заметным дисбалансом
(`rel_plus`: 39, `vital`: 28, `notrel`: 22, `rel_minus`: 16).

- **Accuracy** — общий процент верных оценок. Простая базовая метрика,
  но при дисбалансе классов может быть оптимистичной (модель, угадывающая
  только мажоритарный класс, получит ~37%).
- **Macro F1** — F1, усреднённый по классам без учёта их размера. Главная
  метрика для нас: миноритарный `rel_minus` важен так же, как мажоритарный
  `rel_plus`. Если модель плохо работает на одном из классов, macro F1 это сразу
  покажет.
- **Weighted F1** — F1, взвешенный по размеру класса. Показывает качество
  с точки зрения «средней» строки в выборке.
- **Classification report** — precision/recall/F1 по каждому классу. Помогает
  увидеть, где именно модель ошибается (например, путает `vital` с `rel_plus`).
- **Confusion matrix** — детальная картина ошибок: какие классы между собой
  путаются. На практике частые путаницы будут между соседними оценками
  (`vital` ↔ `rel_plus` и `rel_plus` ↔ `rel_minus`), они менее критичны,
  чем путаница `vital` ↔ `notrel`.

Ориентир для интерпретации: при 4 классах случайное угадывание даёт
accuracy ≈ 0.25 и macro F1 ≈ 0.20–0.25. Целимся хотя бы в macro F1 ≥ 0.55–0.6
для прода — это означает, что промпт сравним с разметкой средненького асессора.


# **<span style="background-color: #fff69d">Версия с CoT (Chain-of-Thought)</span>**

Бейзлайн систематически промахивается на `rel_minus`: модель видит совпадение
по подстроке и сразу ставит `rel_plus`, не проверяя, нет ли более известного
кандидата. Чтобы это исправить — заставим модель **сначала пройти по чек-листу**
и только потом выдать метку.

Подход: тот же чек-лист, что и в бейзлайне, но теперь модель **обязана**
явно ответить на каждый пункт прежде, чем поставить оценку. Это и есть
chain-of-thought — рассуждение перед ответом.

Цена: ответ становится длиннее → больше токенов, больше латентность,
но recall на `rel_minus` должен подняться.


In [10]:
cot_system_prompt = """
Ты — асессор музыкального поискового сервиса. На вход тебе подаются:
- запрос пользователя,
- название найденного трека,
- исполнитель найденного трека.

Твоя задача — выбрать одну метку: vital, rel_plus, rel_minus или notrel.

ОЦЕНКИ:
1) vital — пользователь нашёл именно то, что искал
   (в запросе есть и трек, и исполнитель, оба совпадают).
2) rel_plus — выдача полезная, пользователь доволен.
3) rel_minus — выдача похожа, но есть более вероятный кандидат, который не нашёлся.
4) notrel — выдача совсем не та.

ОБЯЗАТЕЛЬНЫЙ ПОРЯДОК РАССУЖДЕНИЯ:
Прежде чем поставить оценку, ответь КРАТКО (одной фразой каждый) на эти вопросы:

1. Что искал пользователь? Назови предполагаемое намерение
   (известный артист / конкретная песня / жанр / саундтрек / по тексту).

2. Совпадает ли трек из выдачи с этим намерением по смыслу
   (не только по подстроке)?

3. Существует ли более известный одноимённый артист или хит,
   которого пользователь, возможно, имел в виду
   (Вадим Козин, Маша Распутина, Parov Stelar, Земфира и т.п.)?

4. Если в запросе один исполнитель, а в выдаче — коллаб,
   это его известный коллаб или у него есть сольные хиты, которые не нашлись?

5. Если в запросе несколько исполнителей, найдены ли все они вместе?

После ответов на все пять вопросов выведи итоговую метку строго в формате:
Оценка: <vital|rel_plus|rel_minus|notrel>

ПРИМЕРЫ:

Запрос пользователя: цой звезда по имени солнце
Найденный трек: Звезда по имени Солнце
Исполнитель: Виктор Цой
1. Пользователь искал песню «Звезда по имени Солнце» Виктора Цоя.
2. Да, трек совпадает по названию и исполнителю.
3. Нет, это и есть самый известный кандидат.
4. Не применимо — нет коллабов.
5. Не применимо — один исполнитель.
Оценка: vital

Запрос пользователя: включи козина
Найденный трек: Cozinha
Исполнитель: Volbru
1. Пользователь, скорее всего, искал известного исполнителя Вадима Козина.
2. Нет, совпадение только по подстроке «козин»; смысл другой.
3. Да — Вадим Козин, классик романса XX века; выдача про него не знает.
4. Не применимо.
5. Не применимо.
Оценка: rel_minus

Запрос пользователя: включи parov stelar
Найденный трек: Keine Melodien
Исполнитель: Jeans Team, Parov Stelar
1. Пользователь искал электронщика Parov Stelar.
2. Trейк связан с запрошенным исполнителем, но это коллаб.
3. Не применимо.
4. У Parov Stelar много сольных хитов, нашли только его коллаб → rel_minus.
5. Не применимо.
Оценка: rel_minus

Запрос пользователя: включи rammstein москва
Найденный трек: Москва
Исполнитель: Yanix
1. Пользователь искал песню «Moskau» группы Rammstein.
2. Нет, совпадение только по слову «Москва», артист совершенно другой.
3. Нет — Rammstein названы прямо в запросе и не угаданы вообще.
4. Не применимо.
5. Не применимо.
Оценка: notrel

ВАЖНО: после рассуждения последней строкой обязательно поставь
«Оценка: <метка>» — без этой строки ответ не засчитывается.
""".strip()


In [11]:
import os
# Отдельный клиент с увеличенным max_tokens — теперь нужно место под рассуждение
llm_cot = GigaChat(
    credentials=os.environ["GIGACHAT_CREDENTIALS"],  # тот же ключ
    scope="GIGACHAT_API_PERS",
    model="GigaChat-Pro",
    verify_ssl_certs=False,
    streaming=False,
    temperature=0,
    max_tokens=400,  # хватит на 5 коротких пунктов + итоговую строку
)


In [12]:
# Прогон с CoT
cot_raw = []
for _, row in tqdm(test.iterrows(), total=len(test)):
    user_prompt = build_user_prompt(row['query'], row['track_name'], row['author'])
    messages = [
        SystemMessage(content=cot_system_prompt),
        HumanMessage(content=user_prompt),
    ]
    try:
        res = llm_cot.invoke(messages)
        cot_raw.append(res.content)
    except Exception as e:
        cot_raw.append(f"__ERROR__: {e}")

test['raw_prediction_cot'] = cot_raw

# Покажем пример рассуждения, чтобы убедиться что формат соблюдается
print('Пример рассуждения модели:')
print('-' * 60)
print(test['raw_prediction_cot'].iloc[0])
print('-' * 60)


100%|██████████| 105/105 [04:18<00:00,  2.46s/it]

Пример рассуждения модели:
------------------------------------------------------------
1. Пользователь, вероятно, хотел послушать творчество Вадима Козина.
2. Нет, трек не соответствует ожидаемому результату.
3. Да, существует известный исполнитель Вадим Козин, которого мог иметь в виду пользователь.
4. Не применимо.
5. Не применимо.
Оценка: rel_minus
------------------------------------------------------------


In [13]:
# Парсинг ответа с CoT.
# Сначала пробуем найти строку "Оценка: <метка>" в конце ответа.
# Если её нет — падаем на общий parse_label.
import re

def parse_label_cot(raw):
    if not isinstance(raw, str):
        return 'unknown'
    # Ищем явную итоговую строку
    m = re.search(r'Оценк[аи][:\s]+([a-z_]+)', raw, flags=re.IGNORECASE)
    if m:
        candidate = m.group(1).strip().lower()
        if candidate in VALID_LABELS:
            return candidate
    # Fallback: тот же парсер, что для бейзлайна
    return parse_label(raw)

test['prediction_cot'] = test['raw_prediction_cot'].apply(parse_label_cot)
print('Распределение предсказаний (CoT):')
print(test['prediction_cot'].value_counts())
print(f"\nДоля нераспарсенных: {(test['prediction_cot'] == 'unknown').mean():.2%}")


Распределение предсказаний (CoT):
prediction_cot
vital        41
rel_minus    27
rel_plus     25
notrel       12
Name: count, dtype: int64

Доля нераспарсенных: 0.00%


In [14]:
# Сравнение бейзлайн vs CoT
y_true = test['golden_answer']

print('=' * 50)
print('БЕЙЗЛАЙН (T=0, ответ — только метка)')
print('=' * 50)
y_pred_base = test['prediction']
print(f'Accuracy:    {accuracy_score(y_true, y_pred_base):.3f}')
print(f'Macro F1:    {f1_score(y_true, y_pred_base, labels=LABELS, average="macro", zero_division=0):.3f}')
print(f'Weighted F1: {f1_score(y_true, y_pred_base, labels=LABELS, average="weighted", zero_division=0):.3f}')

print('\n' + '=' * 50)
print('CoT (T=0, рассуждение перед меткой)')
print('=' * 50)
y_pred_cot = test['prediction_cot']
print(f'Accuracy:    {accuracy_score(y_true, y_pred_cot):.3f}')
print(f'Macro F1:    {f1_score(y_true, y_pred_cot, labels=LABELS, average="macro", zero_division=0):.3f}')
print(f'Weighted F1: {f1_score(y_true, y_pred_cot, labels=LABELS, average="weighted", zero_division=0):.3f}')

print('\nClassification report (CoT):')
print(classification_report(y_true, y_pred_cot, labels=LABELS, zero_division=0, digits=3))

print('\nConfusion matrix (CoT):')
cm_cot = confusion_matrix(y_true, y_pred_cot, labels=LABELS)
print(pd.DataFrame(cm_cot, index=LABELS, columns=LABELS))

# Главное: что произошло на rel_minus?
print('\n' + '=' * 50)
print('ФОКУС НА rel_minus')
print('=' * 50)
for name, preds in [('бейзлайн', y_pred_base), ('CoT', y_pred_cot)]:
    rm = test[y_true == 'rel_minus']
    correct = (preds[rm.index] == 'rel_minus').sum()
    print(f'{name:10s} | rel_minus correct: {correct}/{len(rm)} '
          f'({correct/len(rm):.0%})')

# Где CoT поменял ответ
changed = test[test['prediction'] != test['prediction_cot']]
print(f'\nИзменено ответов после CoT: {len(changed)} из {len(test)}')
print(f'Из них стало верных: {(changed["prediction_cot"] == changed["golden_answer"]).sum()} '
      f'| было верных: {(changed["prediction"] == changed["golden_answer"]).sum()}')


БЕЙЗЛАЙН (T=0, ответ — только метка)
Accuracy:    0.686
Macro F1:    0.644
Weighted F1: 0.677

CoT (T=0, рассуждение перед меткой)
Accuracy:    0.543
Macro F1:    0.521
Weighted F1: 0.534

Classification report (CoT):
              precision    recall  f1-score   support

       vital      0.585     0.857     0.696        28
    rel_plus      0.640     0.410     0.500        39
   rel_minus      0.333     0.562     0.419        16
      notrel      0.667     0.364     0.471        22

    accuracy                          0.543       105
   macro avg      0.556     0.548     0.521       105
weighted avg      0.584     0.543     0.534       105


Confusion matrix (CoT):
           vital  rel_plus  rel_minus  notrel
vital         24         2          2       0
rel_plus      13        16          9       1
rel_minus      0         4          9       3
notrel         4         3          7       8

ФОКУС НА rel_minus
бейзлайн   | rel_minus correct: 6/16 (38%)
CoT        | rel_minus correc